# Dashboard Analítico – Inflação, Política Monetária e Previsões

Este notebook (NB10) organiza, em formato de “dashboard analítico”:

- IPCA cheio e núcleos
- Difusão, momentum, blocos macroeconômicos
- SELIC, juros reais e Curva de Phillips
- Previsões estatísticas (NB8) e de ML (NB9)

Ideia: ser o “painel executivo” do projeto.

Compatível com Jupytext: 10.dashboard_analitico.py

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

pd.set_option("display.float_format", "{:.4f}".format)
print("PYTHONPATH atualizado:", ROOT)

## 1. Carregar bases consolidadas

In [ ]:
path_selic = os.path.join(ROOT, "data", "processed", "ipca_selic_features.parquet")
path_blocos = os.path.join(ROOT, "data", "processed", "ipca_blocos_macro.parquet")
path_phillips = os.path.join(ROOT, "data", "processed", "curva_phillips_features.parquet")
path_eval_short = os.path.join(ROOT, "data", "processed", "eval_estatistico_ipca_t_plus_1.parquet")
path_eval_med = os.path.join(ROOT, "data", "processed", "eval_estatistico_ipca_12m_ahead.parquet")
path_ml_pred = os.path.join(ROOT, "data", "processed", "ml_predictions.parquet")

df_selic = pd.read_parquet(path_selic)
df_blocos = pd.read_parquet(path_blocos)
df_ph = pd.read_parquet(path_phillips)
df_eval_short = pd.read_parquet(path_eval_short)
df_eval_med = pd.read_parquet(path_eval_med)
df_ml = pd.read_parquet(path_ml_pred)

for df_ in [df_selic, df_blocos, df_ph, df_eval_short, df_eval_med, df_ml]:
    df_.index = pd.to_datetime(df_.index)

df_selic.tail(), df_blocos.tail(), df_ph.tail()

## 2. Painel 1 – IPCA, núcleo e SELIC

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_selic.index, df_selic["ipca"], label="IPCA mensal", color="steelblue", alpha=0.7)
if "nucleo_aparado" in df_ph.columns:
    plt.plot(df_ph.index, df_ph["nucleo_aparado"], label="Núcleo aparado", color="darkorange", linewidth=2)
plt.ylabel("Variação (%)")
plt.title("IPCA cheio vs Núcleo de inflação")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 3))
plt.plot(df_selic.index, df_selic["selic"], label="SELIC", color="darkred")
if "selic_real" in df_selic.columns:
    plt.plot(df_selic.index, df_selic["selic_real"], label="SELIC real", color="purple", linestyle="--")
plt.title("Política monetária – SELIC e juros reais")
plt.ylabel("Taxa (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Painel 2 – Blocos macroeconômicos

In [ ]:
cols_blocos = [c for c in df_blocos.columns if c.startswith("bloco_") and "12m" not in c]
df_blocos_mensal = df_blocos[cols_blocos]

plt.figure(figsize=(12, 5))
df_blocos_mensal.plot(kind="bar", stacked=True, ax=plt.gca(), width=0.9)
plt.title("Contribuição dos blocos macroeconômicos – IPCA mensal (p.p.)")
plt.xlabel("Data")
plt.ylabel("Contribuição (p.p.)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

cols_blocos_12m = [c for c in df_blocos.columns if c.startswith("bloco_12m_")]
df_blocos_12m = df_blocos[cols_blocos_12m]

plt.figure(figsize=(10, 4))
for col in df_blocos_12m.columns:
    plt.plot(df_blocos_12m.index, df_blocos_12m[col], label=col)
plt.title("Contribuição acumulada em 12 meses – Blocos macroeconômicos")
plt.xlabel("Data")
plt.ylabel("Contribuição (p.p. em 12m)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Painel 3 – Curva de Phillips (núcleo vs hiato / juros)

In [ ]:
if "hiato_proxy" in df_ph.columns:
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(df_ph.index, df_ph["nucleo_aparado"], color="steelblue", label="Núcleo aparado")
    ax1.set_ylabel("Núcleo (%)", color="steelblue")
    ax1.tick_params(axis="y", labelcolor="steelblue")

    ax2 = ax1.twinx()
    ax2.plot(df_ph.index, df_ph["hiato_proxy"], color="darkorange", label="Hiato proxy")
    ax2.set_ylabel("Hiato proxy", color="darkorange")
    ax2.tick_params(axis="y", labelcolor="darkorange")

    plt.title("Núcleo de inflação vs Hiato proxy")
    fig.tight_layout()
    plt.grid(True, alpha=0.3)
    plt.show()

if "selic_real" in df_selic.columns:
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(df_ph.index, df_ph["nucleo_aparado"], color="steelblue", label="Núcleo aparado")
    ax1.set_ylabel("Núcleo (%)", color="steelblue")
    ax1.tick_params(axis="y", labelcolor="steelblue")

    ax2 = ax1.twinx()
    ax2.plot(df_selic.index, df_selic["selic_real"], color="purple", label="SELIC real")
    ax2.set_ylabel("SELIC real (%)", color="purple")
    ax2.tick_params(axis="y", labelcolor="purple")

    plt.title("Núcleo de inflação vs SELIC real")
    fig.tight_layout()
    plt.grid(True, alpha=0.3)
    plt.show()

## 5. Painel 4 – Comparação de previsões (estatístico vs ML)

In [ ]:
# Curto prazo – IPCA t+1
df_short = df_eval_short.join(df_ml[["y_test_short", "y_pred_short"]], how="inner")

plt.figure(figsize=(10, 4))
plt.plot(df_short.index, df_short["ipca_t_test"], label="Observado", color="black")
plt.plot(df_short.index, df_short["arima_ipca_t_plus_1"], label="ARIMA", color="steelblue", alpha=0.7)
plt.plot(df_short.index, df_short["arimax_ipca_t_plus_1"], label="ARIMAX", color="darkorange", alpha=0.7)
plt.plot(df_short.index, df_short["y_pred_short"], label="ML (melhor modelo)", color="green", linewidth=2)
plt.title("Previsão de IPCA mensal (t+1) – Estatístico vs ML")
plt.xlabel("Data")
plt.ylabel("IPCA (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Médio prazo – IPCA 12m à frente
df_med = df_eval_med.join(df_ml[["y_test_med", "y_pred_med"]], how="inner")

plt.figure(figsize=(10, 4))
plt.plot(df_med.index, df_med["ipca_12m_t_plus_12_test"], label="Observado", color="black")
plt.plot(df_med.index, df_med["reg_ipca_12m_t_plus_12"], label="Regressão estatística", color="steelblue", alpha=0.7)
plt.plot(df_med.index, df_med["y_pred_med"], label="ML (melhor modelo)", color="darkred", linewidth=2)
plt.title("Previsão de IPCA 12m à frente – Estatístico vs ML")
plt.xlabel("Data")
plt.ylabel("IPCA 12m (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Painel 5 – Erros de previsão

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae

rmse_arima, mae_arima = metrics(df_short["ipca_t_test"], df_short["arima_ipca_t_plus_1"])
rmse_arimax, mae_arimax = metrics(df_short["ipca_t_test"], df_short["arimax_ipca_t_plus_1"])
rmse_ml_short, mae_ml_short = metrics(df_short["ipca_t_test"], df_short["y_pred_short"])

rmse_reg, mae_reg = metrics(df_med["ipca_12m_t_plus_12_test"], df_med["reg_ipca_12m_t_plus_12"])
rmse_ml_med, mae_ml_med = metrics(df_med["ipca_12m_t_plus_12_test"], df_med["y_pred_med"])

df_metrics = pd.DataFrame(
    {
        "Modelo": [
            "ARIMA (t+1)",
            "ARIMAX (t+1)",
            "ML (t+1)",
            "Regressão (12m)",
            "ML (12m)",
        ],
        "RMSE": [
            rmse_arima,
            rmse_arimax,
            rmse_ml_short,
            rmse_reg,
            rmse_ml_med,
        ],
        "MAE": [
            mae_arima,
            mae_arimax,
            mae_ml_short,
            mae_reg,
            mae_ml_med,
        ],
    }
)

df_metrics

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=df_metrics, x="Modelo", y="RMSE", palette="viridis")
plt.title("RMSE por modelo – curto e médio prazo")
plt.xticks(rotation=45)
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Conclusões executivas (para o relatório)

A partir deste dashboard, você consegue escrever, por exemplo:

- A inflação cheia e o núcleo mostram **(aceleração / desaceleração)** recente.
- Os blocos macroeconômicos indicam que a inflação está sendo puxada por:
  - bens industriais / serviços / monitorados / alimentos.
- A política monetária (SELIC real) está:
  - contracionista / expansionista em relação ao núcleo.
- A Curva de Phillips sugere que:
  - o hiato (proxy) está **(positivo / negativo)** e influencia o núcleo.
- Em termos de previsão:
  - modelos de ML superam / empatam / perdem para modelos estatísticos em:
    - curto prazo (t+1)
    - médio prazo (12m)

Este notebook alimenta diretamente o **NB11 – Relatório Final**, onde você transforma
esses gráficos e métricas em narrativa econômica.